# Level 6

In diesem Level geht es um **Persistenz** und **Datenformate**.

In Level 4 hast du gelernt, wie du Dateien liest und schreibst. Dabei haben wir aber nur mit reinem Text gearbeitet. Sobald unser Programm beendet wird, sind alle Werte in unseren Variablen jedoch verloren. In diesem Level lernst du, wie du **strukturierte Daten** (z.B. die Listen und Dictionaries aus Level 3) dauerhaft speicherst und später wieder einliest. Dass Daten über das Programmende hinaus erhalten bleiben, nennt man *Persistenz*.

Wir schauen uns dafür drei verbreitete Möglichkeiten an:

* **CSV** - ein einfaches, tabellarisches Format, das Tabellen abbildet (Zeilen mit Spaltenüberschriften)
* **JSON** - ein Format für verschachtelte Daten, beliebt auch als Schnittstelle zu JavaScript
* **SQLite** - eine richtige [relationale Datenbank](https://de.wikipedia.org/wiki/Relationale_Datenbank) in einer einzigen Datei

Als durchgehendes Beispiel benutzen wir ein kleines Adressbuch. Jeder Kontakt ist ein Dictionary, und alle Kontakte zusammen stehen in einer Liste - das kennst du bereits aus Level 3.

> **Hinweis: nichts zu installieren.** Die drei Module `csv`, `json` und `sqlite3` gehören alle zur **Standardbibliothek** von Python - sie werden mit Python mitgeliefert, ein einfaches `import` genügt also. (SQLite selbst ist dabei bereits eingebaut; das Paket namens `sqlite` auf PyPI brauchst du **nicht** und solltest du nicht installieren.)
>
> Das ist nicht selbstverständlich: Ab Level 12/13 benutzen wir Pakete von Fremdanbietern (z.B. `requests`, `PySide2` oder `flask`), die nicht mitgeliefert werden und erst mit `pip` installiert werden müssen. Solche Pakete installierst du am besten in einer **virtuellen Umgebung** - wie das geht, steht in [Virtuelle Umgebungen.md](../Virtuelle%20Umgebungen.md).

In [ ]:
# Unsere Beispieldaten: eine Liste von Kontakten (jeweils ein dict)
kontakte = [
    {"name": "Max Mustermann", "geburtsjahr": 1982, "email": "max@example.org"},
    {"name": "Martha Musterfrau", "geburtsjahr": 2001, "email": "martha@example.org"},
    {"name": "Erika Beispiel", "geburtsjahr": 1987, "email": "erika@example.org"},
]
for kontakt in kontakte:
    print(kontakt)

## CSV

**CSV** steht für *Comma-Separated Values*, also "durch Kommata getrennte Werte". Es ist ein sehr einfaches, tabellarisches Format: Jede Zeile der Datei ist ein Datensatz (eine "Zeile der Tabelle"), und die einzelnen Felder (die "Spalten") werden durch ein Trennzeichen - meistens ein Komma - voneinander getrennt. Programme wie Excel oder LibreOffice Calc können CSV-Dateien direkt öffnen.

In Level 5 haben wir gesehen, dass man mit `str.split(";")` einen einfachen CSV-Parser bauen kann. Das ist aber fehleranfällig: Sobald ein Feld selbst das Trennzeichen enthält (z.B. ein Komma im Namen) oder über mehrere Zeilen geht, funktioniert das nicht mehr richtig. Deshalb bringt Python das Modul [`csv`](https://docs.python.org/3/library/csv.html) mit, das uns diese Arbeit abnimmt.

In [ ]:
import csv
from pathlib import Path

csv_datei = Path("kontakte.csv")

# newline="" ist beim Schreiben von CSV-Dateien empfohlen,
# damit die Zeilenumbrüche korrekt gesetzt werden.
with csv_datei.open("w", newline="", encoding="utf-8") as datei:
    writer = csv.writer(datei)
    writer.writerow(["name", "geburtsjahr", "email"])  # die Kopfzeile, in der Regel Spaltenüberschriften
    for kontakt in kontakte:
        writer.writerow([kontakt["name"], kontakt["geburtsjahr"], kontakt["email"]])

# Schauen wir uns an, was in der Datei gelandet ist:
print(csv_datei.read_text(encoding="utf-8"))

Zum Lesen benutzen wir `csv.reader`. Über das zurückgegebene Objekt können wir wie über eine Liste iterieren - jede Zeile ist dabei eine Liste von Strings.

In [ ]:
with csv_datei.open("r", newline="", encoding="utf-8") as datei:
    reader = csv.reader(datei)
    for zeile in reader:
        print(zeile)

Noch bequemer wird es mit `csv.DictWriter` und `csv.DictReader`. Diese arbeiten direkt mit Dictionaries und benutzen die Spaltennamen als Schlüssel - das passt perfekt zu unseren Kontakten.

In [ ]:
with csv_datei.open("w", newline="", encoding="utf-8") as datei:
    writer = csv.DictWriter(datei, fieldnames=["name", "geburtsjahr", "email"])
    writer.writeheader()          # schreibt die Kopfzeile automatisch
    writer.writerows(kontakte)    # schreibt gleich alle Dictionaries

print(csv_datei.read_text(encoding="utf-8"))

In [ ]:
with csv_datei.open("r", newline="", encoding="utf-8") as datei:
    reader = csv.DictReader(datei)
    gelesen = list(reader)

for kontakt in gelesen:
    print(kontakt)

Beachte eine wichtige Eigenschaft von CSV: In der Datei steht **nur Text**. Beim Einlesen bekommen wir daher jeden Wert als `string` zurück - auch das Geburtsjahr, das wir als `int` geschrieben haben. Wenn wir damit rechnen wollen, müssen wir es erst wieder mit `int()` umwandeln.

In [ ]:
print(gelesen[0]["geburtsjahr"])
print(type(gelesen[0]["geburtsjahr"]))

# Zum Rechnen zurück in einen integer umwandeln:
print(2026 - int(gelesen[0]["geburtsjahr"]), "Jahre alt")

Das Trennzeichen lässt sich übrigens frei wählen. Im deutschsprachigen Raum ist das Semikolon `;` verbreitet, weil im Komma oft das Dezimaltrennzeichen steckt. Dazu geben wir `delimiter` an:

In [ ]:
with csv_datei.open("w", newline="", encoding="utf-8") as datei:
    writer = csv.writer(datei, delimiter=";")
    writer.writerow(["name", "geburtsjahr", "email"])
    writer.writerows([[k["name"], k["geburtsjahr"], k["email"]] for k in kontakte])

print(csv_datei.read_text(encoding="utf-8"))

## JSON

CSV eignet sich gut für flache Tabellen. Sobald unsere Daten aber **verschachtelt** sind (ein Kontakt hat z.B. mehrere Telefonnummern als Liste), stößt CSV an seine Grenzen.

**JSON** (*JavaScript Object Notation*) ist ein Format, das genau dafür gemacht ist. Es kann Listen und Dictionaries beliebig tief ineinander verschachteln und ist trotzdem gut menschenlesbar. JSON ist heute das Standardformat für den Datenaustausch im Internet (fast jede Web-Schnittstelle liefert JSON) und wird auch gerne für Konfigurationsdateien benutzt. Die Struktur von JSON ähnelt nicht zufällig den Python-Literalen für Listen und Dictionaries.

Python bringt dafür das Modul [`json`](https://docs.python.org/3/library/json.html) mit. Mit `json.dumps()` (für *dump string*) verwandeln wir ein Python-Objekt in einen JSON-String.

In [ ]:
import json

# indent sorgt für eine eingerückte, lesbare Darstellung.
# ensure_ascii=False lässt Umlaute als Umlaute stehen (statt als \uXXXX).
text = json.dumps(kontakte, indent=2, ensure_ascii=False)
print(text)

Die umgekehrte Richtung übernimmt `json.loads()` (für *load string*): Aus einem JSON-String wird wieder ein Python-Objekt. Anders als bei CSV bleiben dabei die **Typen erhalten** - das Geburtsjahr ist nach dem Einlesen wieder ein `int`.

In [ ]:
wieder_eingelesen = json.loads(text)
print(wieder_eingelesen[0])
print(type(wieder_eingelesen[0]["geburtsjahr"]))

Um direkt mit Dateien zu arbeiten, gibt es `json.dump()` und `json.load()` (jeweils ohne das `s`). Sie bekommen zusätzlich das offene Datei-Objekt übergeben.

In [ ]:
json_datei = Path("kontakte.json")

# Schreiben:
with json_datei.open("w", encoding="utf-8") as datei:
    json.dump(kontakte, datei, indent=2, ensure_ascii=False)

# Lesen:
with json_datei.open("r", encoding="utf-8") as datei:
    geladen = json.load(datei)

print(geladen)

Zwei Stolpersteine solltest du kennen:

1. **Schlüssel werden immer zu Strings.** JSON kennt als Schlüssel nur Strings. Ein `dict` mit integer-Schlüsseln lässt sich zwar speichern, beim Einlesen sind die Schlüssel aber Strings.
2. **Nicht jeder Typ lässt sich direkt speichern.** `list`, `dict`, `str`, `int`, `float`, `bool` und `None` funktionieren sofort. Andere Typen musst du vorher in einen passenden Typ umwandeln, sonst gibt es einen `TypeError`. Das ist aber meist einfach (siehe unten).

In [ ]:
# Der integer-Schlüssel 1 wird beim Speichern zum String "1":
print(json.dumps({1: "eins", 2: "zwei"}))

Für die häufigsten dieser Fälle gibt es einfache Auswege:

* Ein **`tuple`** wird beim Speichern automatisch zu einer JSON-Liste (JSON kennt keine Tupel).
* Ein **`set`** hat in JSON keine Entsprechung - wir wandeln es vorher selbst mit `list()` in eine Liste um.
* Ein **Datum** (`datetime`) speichern wir als Text im standardisierten ISO-Format mit `.isoformat()`.

In [ ]:
from datetime import datetime

# Ein tuple wird automatisch zu einer JSON-Liste:
print(json.dumps((1, 2, 3)))

# Ein set wandeln wir vorher selbst in eine Liste um:
zahlen = {1, 2, 3}
print(json.dumps(list(zahlen)))

# Ein datetime speichern wir als Text im ISO-Format:
jetzt = datetime(2024, 3, 14, 15, 9, 26)
print(json.dumps(jetzt.isoformat()))

In [ ]:
# Beim Einlesen bekommen wir den Text zurück und können ihn
# mit datetime.fromisoformat() wieder in ein Datum verwandeln:
text = json.dumps(jetzt.isoformat())
wieder_datum = datetime.fromisoformat(json.loads(text))
print(wieder_datum)
print(type(wieder_datum))

## SQLite

CSV und JSON laden immer die **gesamte** Datei. Solange wir wenige Daten haben, ist das kein Problem. Wollen wir aber aus sehr vielen Datensätzen gezielt einzelne suchen, ändern oder löschen, wird das schnell unpraktisch. Dafür gibt es **Datenbanken**.

[SQLite](https://www.sqlite.org/) ist eine besonders einfache Datenbank: Sie braucht keinen Server, sondern speichert alles in einer einzigen Datei. Python bringt sie über das Modul [`sqlite3`](https://docs.python.org/3/library/sqlite3.html) bereits mit. Abgefragt werden Datenbanken mit der Sprache **[SQL](https://de.wikipedia.org/wiki/SQL)** - die schauen wir uns hier nur ganz kurz an.

Zuerst stellen wir eine Verbindung zur Datenbank-Datei her und legen über einen *Cursor* eine Tabelle an. Eine Tabelle hat benannte Spalten mit einem festen Typ.

In [ ]:
import sqlite3

db_datei = Path("kontakte.db")
if db_datei.exists():
    db_datei.unlink()  # für einen sauberen Start das evtl. alte Beispiel löschen

verbindung = sqlite3.connect(db_datei)
cursor = verbindung.cursor()

cursor.execute(
    "CREATE TABLE kontakte (name TEXT, geburtsjahr INTEGER, email TEXT)"
)

Jetzt füllen wir die Tabelle. **Dabei ist eine Regel sehr wichtig**:

> ### WARNUNG: Niemals Werte in den SQL-Text einbauen!
> Setze Werte **nie** per String-Verkettung oder f-string in den Befehl ein. Benutze stattdessen Fragezeichen `?` als Platzhalter und übergib die Werte als zweites Argument. Python setzt sie dann sicher ein. Andernfalls öffnest du die Tür für die sogenannte *SQL-Injection*, eine der häufigsten Sicherheitslücken überhaupt.

Mit `executemany()` können wir gleich mehrere Datensätze auf einmal einfügen. Am Ende müssen wir mit `commit()` bestätigen, damit die Änderungen tatsächlich in die Datei geschrieben werden.

In [ ]:
# Die Werte als Liste von Tupeln vorbereiten:
daten = [(k["name"], k["geburtsjahr"], k["email"]) for k in kontakte]

# Die ? sind Platzhalter, die sicher durch die Werte ersetzt werden:
cursor.executemany("INSERT INTO kontakte VALUES (?, ?, ?)", daten)

verbindung.commit()  # Änderungen dauerhaft speichern

Daten holen wir mit `SELECT`. Über den Cursor können wir anschließend mit `fetchall()` alle Treffer als Liste von Tupeln bekommen (oder mit `fetchone()` nur den nächsten).

In [ ]:
cursor.execute("SELECT * FROM kontakte")
for zeile in cursor.fetchall():
    print(zeile)

Der große Vorteil einer Datenbank ist, dass wir gezielt abfragen können, ohne alle Daten selbst durchzugehen. Mit `WHERE` schränken wir die Treffer ein - und auch hier benutzen wir wieder den Platzhalter `?` für den Wert.

In [ ]:
# Alle Kontakte, die nach 1990 geboren wurden:
cursor.execute("SELECT name, email FROM kontakte WHERE geburtsjahr > ?", (1990,))
for zeile in cursor.fetchall():
    print(zeile)

Zum Schluss schließen wir die Verbindung wieder mit `close()`.

In [ ]:
verbindung.close()

## Rückblick

In diesem Level hast du drei Wege kennengelernt, Daten dauerhaft zu speichern:

* **CSV** eignet sich für einfache, tabellarische Daten. Es ist gut mit Tabellenprogrammen kompatibel, kennt aber nur Text (alles kommt als `string` zurück) und keine Verschachtelung.
* **JSON** eignet sich für verschachtelte Daten, erhält die Typen und ist der Standard für den Datenaustausch im Internet.
* **SQLite** ist eine vollwertige Datenbank in einer Datei. Sie lohnt sich, wenn du in vielen Datensätzen gezielt suchen, ändern oder löschen möchtest.

Welches Format das richtige ist, hängt vom Anwendungsfall ab - für ein kleines Adressbuch tut es JSON, für eine große, durchsuchbare Sammlung ist eine Datenbank die bessere Wahl.

In [ ]:
# Aufräumen: die in diesem Level erzeugten Dateien wieder entfernen
for datei in [csv_datei, json_datei, db_datei]:
    if datei.exists():
        datei.unlink()